In [6]:
%cd /home/parthgandhi/Projects/MLBot/mlbot

/home/parthgandhi/Projects/MLBot/mlbot


In [66]:
import polars as pl
from src.swing_model.features import basic_features, moving_average_features
import numpy as np
from datetime import datetime

In [54]:
def rolling_slope(s: pl.Series) -> float:
    arr = s.to_numpy()

    if len(arr) < 2 or np.isnan(arr).any():
        return np.nan

    x = np.arange(len(arr), dtype=np.float64)

    x_mean = x.mean()
    y_mean = arr.mean()

    cov = ((x - x_mean) * (arr - y_mean)).sum()
    var = ((x - x_mean) ** 2).sum()

    return cov / var if var > 0 else np.nan


def rolling_r2(s: pl.Series) -> float:
    arr = s.to_numpy()

    if len(arr) < 2 or np.isnan(arr).any():
        return np.nan

    x = np.arange(len(arr), dtype=np.float64)

    x_mean = x.mean()
    y_mean = arr.mean()

    cov = ((x - x_mean) * (arr - y_mean)).sum()

    var_x = ((x - x_mean) ** 2).sum()
    var_y = ((arr - y_mean) ** 2).sum()

    if var_x <= 0 or var_y <= 0:
        return np.nan

    return (cov**2) / (var_x * var_y)

In [53]:
data = pl.scan_parquet("src/test_data.parquet")

In [14]:
# res = basic_features(data=data)
# res = moving_average_features(data=res)

In [15]:
# res.filter(pl.col("symbol").is_in(["ITC"])).remove(
#     pl.any_horizontal(pl.col("*").is_null())
# ).collect()

In [ ]:
(
    data.with_columns(pl.col("timestamp").cast(pl.Date()))
    .drop("volume")
    .with_columns(
        # Prev Price Columns
        [
            pl.col(col)
            .shift(1)
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .alias(f"prev_{col}")
            for col in ["open", "high", "low", "close"]
        ]
        # Close SMA expression
        + [
            pl.col(col)
            .rolling_mean(window_size=n)
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .round(2)
            .alias(f"{col}_sma_{n}")
            for n in [50, 200]
            for col in ["close"]
        ]
        # Close EMA Experssion
        + [
            pl.col("close")
            .ewm_mean(alpha=2 / (n + 1))
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .round(2)
            .alias(f"close_ema_{n}")
            for n in [9, 21]
        ]
        # 52 week high
        + [
            pl.col("close")
            .rolling_max(window_size=252)
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .alias("high_52W")
        ]
        # 52 week low
        + [
            pl.col("close")
            .rolling_min(window_size=252)
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .alias("low_52W")
        ]
        # Candle Red or Green
        + [
            pl.when(pl.col("close") > pl.col("open"))
            .then(True)
            .otherwise(False)
            .alias("is_green_candle")
        ]
        # Range of Candle
        + [(pl.col("high") - pl.col("low")).round(2).alias("range")]
        # Body of Canlde
        + [(pl.col("open") - pl.col("close")).abs().round(2).alias("body")]
        # Upper Wick
        + [
            (pl.col("high") - pl.max_horizontal("open", "close"))
            .round(2)
            .alias("upper_wick")
        ]
        # Lower Wick
        + [
            (pl.min_horizontal("open", "close") - pl.col("low"))
            .round(2)
            .alias("lower_wick")
        ]
        # Day Range
        + [(pl.col("high") / pl.col("low")).round(4).alias("day_range")]
        # Close Position in Range
        + [
            ((pl.col("close") - pl.col("low")) / (pl.col("high") - pl.col("low")))
            .round(4)
            .alias("close_position_in_range")
        ]
        # ROC over Short Time Period
        + [
            (pl.col("close") / pl.col("close").shift(n) - 1)
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .round(4)
            .alias(f"roc_{n}")
            for n in [3, 5, 10]
        ]
    )
    .with_columns(
        # Log Returns Calculated for Changes
        [
            (pl.col(col) / pl.col(f"prev_{col}"))
            .log()
            .round(4)
            .alias(f"log_return_{col}")
            for col in ["open", "high", "low", "close"]
        ]
        # Percentage off 52 week high
        + [
            ((pl.col("close") / pl.col("high_52W")) - 1)
            .clip(upper_bound=1)
            .round(4)
            .alias("dst_from_high_52W")
        ]
        # True Range Calculation for ATR
        + [
            pl.max_horizontal(
                pl.col("high") - pl.col("low"),
                (pl.col("high") - pl.col("prev_close")).abs(),
                (pl.col("low") - pl.col("prev_close")).abs(),
            ).alias("true_range")
        ]
        # Calculate Standard Deviation based on Close, EMA9 and EMA21
        + [
            pl.concat_list("close_ema_9", "close_ema_21", "close")
            .list.std(ddof=0)
            .round(4)
            .alias("std_9_21")
        ]
        # Calculate Standard Deviation based on Close, EMA9, EMA21 and SMA50
        + [
            pl.concat_list("close_ema_9", "close_ema_21", "close", "close_sma_50")
            .list.std(ddof=0)
            .round(4)
            .alias("std_9_21_50")
        ]
        # Body, Upper Wick and Lower Wick Pct to Range
        + [
            (pl.col(col) / pl.col("range"))
            .round(4)
            .clip(upper_bound=1, lower_bound=0)
            .alias(f"{col}_to_range_pct")
            for col in ["body", "upper_wick", "lower_wick"]
        ]
        # Distance Between SMA 50 and SMA 200
        + [
            ((pl.col(col) / pl.col("close_sma_200")) - 1)
            .round(4)
            .alias(f"{col}_dist_from_close_sma_200")
            for col in ["close", "close_sma_50"]
        ]
        # SMA50 GTE SMA200
        + [
            pl.when(pl.col("close_sma_50") >= pl.col("close_sma_200"))
            .then(True)
            .otherwise(False)
            .alias("is_sma_50_gte_sma_200")
        ]
        # Distance of Close and Low to Moving Averages
        + [
            (((pl.col(m_col) / pl.col(c_col)) - 1).round(4)).alias(
                f"{m_col}_dist_from_{c_col}"
            )
            for m_col in ["close", "low"]
            for c_col in ["close_ema_9", "close_ema_21", "close_sma_50"]
        ]
        # Calculate ADR 20
        + [
            pl.col("day_range")
            .rolling_mean(window_size=n)
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .round(4)
            .alias(f"adr_{n}")
            for n in [20]
        ]
        # MA Alignment fot Bullish
        + [
            pl.when(
                (pl.col("close_ema_9") >= pl.col("close_sma_50"))
                | (pl.col("close_ema_21") >= pl.col("close_sma_50"))
            )
            .then(True)
            .otherwise(False)
            .alias("ma_aligned_bullish")
        ]
    )
    .with_columns(
        # Calculate ATR using True Range
        [
            pl.col("true_range")
            .ewm_mean(alpha=2 / (n + 1))
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .round(4)
            .alias(f"atr_{n}")
            for n in [14, 20, 50]
        ]
        # Calculate the smooth dispersion score based on the normalzied standard deviation
        + [
            pl.col(col)
            .rolling_mean(window_size=n)
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .round(4)
            .alias(f"{col}_dispersion_{n}")
            for n in [5, 10, 15, 20]
            for col in ["std_9_21", "std_9_21_50"]
        ]
    )
    .with_columns(
        # ATR ratio 14 by 50
        [(pl.col("atr_14") / pl.col("atr_50")).round(4).alias("atr_ratio_14_50")]
        # ATR 20 Pct
        + [(pl.col("atr_20") / pl.col("close")).round(4).alias("atr_pct_20")]
        # ADR 20 Pct
        + [(pl.col("adr_20") / pl.col("close")).round(4).alias("adr_pct_20")]
    )
    .with_columns(
        [
            pl.col(col)
            .log()
            .rolling_map(rolling_slope, window_size=n)
            .over(
                partition_by="symbol",
                order_by="timestamp",
                descending=False,
            )
            .round(4)
            .alias(f"{col}_regression_slope_{n}")
            for n in [3, 5, 10]
            for col in ["close", "close_ema_21", "close_sma_50"]
        ]
        + [
            pl.col(col)
            .rolling_map(rolling_r2, window_size=n)
            .over(
                partition_by="symbol",
                order_by="timestamp",
                descending=False,
            )
            .round(4)
            .alias(f"{col}_regression_r2_{n}")
            for n in [3, 5, 10]
            for col in ["close", "close_ema_21", "close_sma_50"]
        ]
    )
    .select(
        [
            "symbol",
            "timestamp",
            "is_green_candle",
            "close_position_in_range",
            "roc_3",
            "roc_5",
            "roc_10",
            "log_return_open",
            "log_return_high",
            "log_return_low",
            "log_return_close",
            "dst_from_high_52W",
            "std_9_21",
            "std_9_21_50",
            "body_to_range_pct",
            "upper_wick_to_range_pct",
            "lower_wick_to_range_pct",
            "close_dist_from_close_sma_200",
            "close_sma_50_dist_from_close_sma_200",
            "is_sma_50_gte_sma_200",
            "close_dist_from_close_ema_9",
            "close_dist_from_close_ema_21",
            "close_dist_from_close_sma_50",
            "low_dist_from_close_ema_9",
            "low_dist_from_close_ema_21",
            "low_dist_from_close_sma_50",
            "ma_aligned_bullish",
            "std_9_21_dispersion_5",
            "std_9_21_50_dispersion_5",
            "std_9_21_dispersion_10",
            "std_9_21_50_dispersion_10",
            "std_9_21_dispersion_15",
            "std_9_21_50_dispersion_15",
            "std_9_21_dispersion_20",
            "std_9_21_50_dispersion_20",
            "atr_ratio_14_50",
            "atr_pct_20",
            "adr_pct_20",
            "close_regression_slope_3",
            "close_ema_21_regression_slope_3",
            "close_sma_50_regression_slope_3",
            "close_regression_slope_5",
            "close_ema_21_regression_slope_5",
            "close_sma_50_regression_slope_5",
            "close_regression_slope_10",
            "close_ema_21_regression_slope_10",
            "close_sma_50_regression_slope_10",
            "close_regression_r2_3",
            "close_ema_21_regression_r2_3",
            "close_sma_50_regression_r2_3",
            "close_regression_r2_5",
            "close_ema_21_regression_r2_5",
            "close_sma_50_regression_r2_5",
            "close_regression_r2_10",
            "close_ema_21_regression_r2_10",
            "close_sma_50_regression_r2_10",
        ]
    )
).filter(pl.col("symbol") == "TDPOWERSYS").filter(
    pl.col("timestamp") <= datetime(2026, 3, 16)
).collect()

symbol,timestamp,is_green_candle,close_position_in_range,roc_3,roc_5,roc_10,log_return_open,log_return_high,log_return_low,log_return_close,dst_from_high_52W,std_9_21,std_9_21_50,body_to_range_pct,upper_wick_to_range_pct,lower_wick_to_range_pct,close_dist_from_close_sma_200,close_sma_50_dist_from_close_sma_200,is_sma_50_gte_sma_200,close_dist_from_close_ema_9,close_dist_from_close_ema_21,close_dist_from_close_sma_50,low_dist_from_close_ema_9,low_dist_from_close_ema_21,low_dist_from_close_sma_50,ma_aligned_bullish,std_9_21_dispersion_5,std_9_21_50_dispersion_5,std_9_21_dispersion_10,std_9_21_50_dispersion_10,std_9_21_dispersion_15,std_9_21_50_dispersion_15,std_9_21_dispersion_20,std_9_21_50_dispersion_20,atr_ratio_14_50,atr_pct_20,adr_pct_20,close_regression_slope_3,close_ema_21_regression_slope_3,close_sma_50_regression_slope_3,close_regression_slope_5,close_ema_21_regression_slope_5,close_sma_50_regression_slope_5,close_regression_slope_10,close_ema_21_regression_slope_10,close_sma_50_regression_slope_10,close_regression_r2_3,close_ema_21_regression_r2_3,close_sma_50_regression_r2_3,close_regression_r2_5,close_ema_21_regression_r2_5,close_sma_50_regression_r2_5,close_regression_r2_10,close_ema_21_regression_r2_10,close_sma_50_regression_r2_10
str,date,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,f64,f64,f64,f64,f64,f64,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""TDPOWERSYS""",2024-10-24,false,0.3416,null,null,null,null,null,null,null,null,0.0,0.0,0.4074,0.251,0.3416,null,null,false,0.0,0.0,null,-0.0105,-0.0105,null,false,null,null,null,null,null,null,null,null,1.0,0.0308,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""TDPOWERSYS""",2024-10-25,false,0.3504,null,null,null,-0.0125,-0.0152,-0.0427,-0.0325,null,2.739,2.739,0.5625,0.0871,0.3504,null,null,false,-0.0145,-0.0155,null,-0.0347,-0.0357,null,false,null,null,null,null,null,null,null,null,1.0152,0.0459,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""TDPOWERSYS""",2024-10-28,false,0.5136,null,null,null,-0.0366,-0.0132,-0.018,-0.0059,null,2.3369,2.3369,0.0294,0.457,0.5136,null,null,false,-0.0121,-0.0136,null,-0.044,-0.0455,null,false,null,null,null,null,null,null,null,null,1.0199,0.0523,null,-0.0192,-0.0124,null,null,null,null,null,null,null,0.8606,0.8606,0.8606,null,null,null,null,null,null
"""TDPOWERSYS""",2024-10-29,false,0.1763,-0.0392,null,null,0.0154,0.0141,0.0212,-0.0016,null,1.8186,1.8186,0.3341,0.4896,0.1763,null,null,false,-0.0091,-0.0109,null,-0.019,-0.0208,null,false,null,null,null,null,null,null,null,null,1.0174,0.0537,null,-0.0037,-0.0061,null,null,null,null,null,null,null,0.8995,0.8995,0.8995,null,null,null,null,null,null
"""TDPOWERSYS""",2024-10-30,false,0.467,0.0295,null,null,0.0309,0.0161,0.0239,0.0366,null,3.5428,3.5428,0.2744,0.2586,0.467,null,null,false,0.0194,0.0196,null,-0.0036,-0.0033,null,false,2.0875,2.0875,null,null,null,null,null,null,1.0209,0.0542,null,0.0175,0.0009,null,-0.0014,-0.0058,null,null,null,null,0.7167,0.7167,0.7167,0.0135,0.0135,0.0135,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""TDPOWERSYS""",2026-03-10,false,0.0911,-0.078,-0.0837,-0.0979,0.0043,0.0034,-0.0226,-0.0362,-0.0979,19.6956,36.3407,0.7427,0.1662,0.0911,0.283,0.2086,true,-0.0542,-0.0397,0.0616,-0.0598,-0.0455,0.0552,true,14.8857,44.912,19.0987,50.1383,23.8299,52.8349,27.3573,52.8477,1.0921,0.0514,0.0013,-0.0374,-0.0024,0.003,-0.019,0.0007,0.0034,-0.0069,0.0038,0.0043,0.9991,0.9991,0.9991,0.759,0.759,0.759,0.4976,0.4976,0.4976
"""TDPOWERSYS""",2026-03-11,false,0.1045,-0.0866,-0.0842,-0.0963,-0.0453,-0.0249,-0.0155,-0.0157,-0.1119,21.2706,32.6744,0.3225,0.573,0.1045,0.2599,0.2091,true,-0.0559,-0.05,0.0421,-0.0614,-0.0555,0.036,true,16.1684,41.5466,18.5049,47.6546,22.9702,51.2848,26.1457,51